# 사전준비

# 1. 모델 불러오기

## 1) 베이스 모델 불러오기

In [ ]:
%pip install -U unsloth trl transformers accelerate bitsandbytes datasets


In [ ]:
import os, tempfile
from unsloth import FastLanguageModel
import torch

max_seq_length =
dtype = None
load_in_4bit =

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name =     ,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit
)

print(f"모델 로드 완료!")
print(f"   dtype : {next(model.parameters()).dtype}")
print(f"   device: {next(model.parameters()).device}")


## 2) LoRA 어댑터 준비

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r =     ,
    target_modules = [
            ,
    ],
    lora_alpha =     ,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print("LoRA 어댑터 추가 완료!")


# 2. 데이터셋 불러오기

## 1) 데이터셋 로드

## 2) formatting_func 만들기

In [ ]:
def formatting_prompts_func(data):
    instructions = data[    ]
    outputs = data[    ]

    result = []
    for instruction, output in zip(instructions, outputs):
        messages = [
            {"role":     , "content": },
            {"role":     , "content": },
        ]
        chat_message = tokenizer.apply_chat_template(
            messages,
            tokenize =     ,
            add_generation_prompt =
        )
        result.append(chat_message)

    return


In [ ]:
# Trainer를 만들 때 데이터가 이렇게 정돈됩니다.
data = {
    "instruction": [
        '바이든 대통령이 발표한 행정명령에서 AI 시스템의 안전성과 신뢰성을 확인하기 위해 어떤 조치를 추진하고 있습니까?',
        'G7 국가들이 채택한 AI 국제 행동강령에 따르면, 첨단 AI 시스템의 개발 과정에서 어떤 조치를 취해야 합니까?'
    ],
    "output": [
        '바이든 대통령이 발표한 행정명령에서는 강력한 AI 시스템을 개발하는 기업에게 안전 테스트 결과와 시스템에 관한 주요 정보를 미국 정부와 공유할 것을 요구하고, AI 시스템의 안전성과 신뢰성 확인을 위한 표준 및 AI 생성 콘텐츠 표시를 위한 표준과 모범사례 확립을 추진하고 있습니다.',
        'G7 국가들은 첨단 AI 시스템의 개발 과정에서 AI 수명주기 전반에 걸쳐 위험을 평가 및 완화하는 조치를 채택해야 합니다.'
    ]
}


In [ ]:
formatting_prompts_func(data)

In [ ]:
## 데이터셋 변환 과정
# 데이터셋 원본
{
    'instruction': '바이든 대통령이 발표한 행정명령에서 AI 시스템의 안전성과 신뢰성을 확인하기 위해 어떤 조치를 추진하고 있습니까?',
    'input': '',
    'output': '바이든 대통령이 발표한 행정명령에서는 강력한 AI 시스템을 개발하는 기업에게 안전 테스트 결과와 시스템에 관한 주요 정보를 미국 정부와 공유할 것을 요구하고, AI 시스템의 안전성과 신뢰성 확인을 위한 표준 및 AI 생성 콘텐츠 표시를 위한 표준과 모범사례 확립을 추진하고 있습니다.'
}

# chat형식의 messages로 바꾸기
[
    {"role": "user", "content": '바이든 대통령이 발표한 행정명령에서 AI 시스템의 안전성과 신뢰성을 확인하기 위해 어떤 조치를 추진하고 있습니까?'},
    {"role": "assistant", "content": '바이든 대통령이 발표한 행정명령에서는 강력한 AI 시스템을 개발하는 기업에게 안전 테스트 결과와 시스템에 관한 주요 정보를 미국 정부와 공유할 것을 요구하고, AI 시스템의 안전성과 신뢰성 확인을 위한 표준 및 AI 생성 콘텐츠 표시를 위한 표준과 모범사례 확립을 추진하고 있습니다.'},
]

# 숫자 벡터로 바꾸려면 messages를 하나의 문자열로 변환해야 한다. -- apply_chat_template
'<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\n바이든 대통령이 발표한 행정명령에서 AI 시스템의 안전성과 신뢰성을 확인하기 위해 어떤 조치를 추진하고 있습니까?<|im_end|>\n<|im_start|>assistant\n바이든 대통령이 발표한 행정명령에서는 강력한 AI 시스템을 개발하는 기업에게 안전 테스트 결과와 시스템에 관한 주요 정보를 미국 정부와 공유할 것을 요구하고, AI 시스템의 안전성과 신뢰성 확인을 위한 표준 및 AI 생성 콘텐츠 표시를 위한 표준과 모범사례 확립을 추진하고 있습니다.<|im_end|>\n'

# 3. Trainer 만들기

## 1) TrainingArgument

In [ ]:
from trl import SFTConfig
from unsloth import is_bfloat16_supported

args = SFTConfig(
    per_device_train_batch_size =     ,
    gradient_accumulation_steps =     ,
    warmup_steps =     ,
    max_steps =     ,
    learning_rate =     ,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir =     ,
    save_strategy = "no",
    report_to = "none",
    max_length = max_seq_length,
    dataset_num_proc = None,
    packing = False,
    average_tokens_across_devices = False,
)


## 2) SFTTrainer

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset =     ,
    formatting_func =     ,
    args = args,
)

print("트레이너 설정 완료!")


# 4. 학습

## 1) 학습 전 GPU 현황

In [ ]:
# 학습 전 GPU 메모리 상태 기록 (학습 후 셀과 비교용)
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU 이름     : {gpu_stats.name}")
print(f"전체 VRAM    : {max_memory} GB")
print(f"현재 예약량  : {start_gpu_memory} GB")
print(f"남은 여유    : {round(max_memory - start_gpu_memory, 3)} GB")


## 2) 학습하기

In [ ]:
trainer_stats = trainer.train()

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print(f"학습 완료!")
print(f"")
print(f"학습 시간        : {round(trainer_stats.metrics['train_runtime'] / 60, 2)} 분")
print(f"")
print(f"전체 VRAM 사용량 : {used_memory} GB ({used_percentage} %)")
print(f"LoRA 학습 사용량 : {used_memory_for_lora} GB ({lora_percentage} %)")
print(f"   (모델 로드 제외, 순수 학습에 쓴 VRAM)")


## 3) 모델 저장

In [ ]:
model.save_pretrained_merged(
        ,
    tokenizer,
    save_method =
)

print("Merged model saved.")
print("Run the next cell to restart the Colab runtime, then continue from section 5.")


In [ ]:
import os
import time

print("Restarting runtime. After reconnecting, continue from section 5.")
time.sleep(2)
os.kill(os.getpid(), 9)


# 5. 추론

## 1) 모델 불러온 후 추론

In [ ]:
import sys

if any(name == "unsloth" or name.startswith("unsloth.") for name in sys.modules):
    raise RuntimeError(
        "Unsloth is still loaded in this runtime. "
        "Restart the Colab runtime, then run only section 5 for inference."
    )

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path =

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype = torch.bfloat16,
    device_map = "auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_path)

print("Model loaded.")


In [ ]:
input_text = tokenizer.apply_chat_template(
    [{"role":     , "content":     }],
    tokenize =     ,
    add_generation_prompt =
)
inputs = tokenizer(input_text, return_tensors="pt").to(    )
print(inputs)


In [ ]:
from transformers import TextStreamer

print("=== After fine-tuning ===")
text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens =     ,
    use_cache = False,
    repetition_penalty =     ,
    temperature =     ,
    do_sample =     ,
)


# 2) Ollama에 내 모델 등록하기

## 1) Unsloth로 모델 불러오기

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length =
dtype = None
load_in_4bit =

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name =     ,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit
)

print(f"모델 로드 완료!")


## 2) GGUF로 모델 저장하기

In [ ]:
model.save_pretrained_gguf(
        ,
    tokenizer,
    quantization_method =
)


In [ ]:
model.save_pretrained_gguf(
        ,
    tokenizer,
    quantization_method =
)


## 3) Ollama에 내 모델 GGUF 등록하기

In [ ]:
# 터미널에서 실행하세요
# cd models/model_merged_gguf
# ollama create 내모델이름설정 -f Modelfile
# ollama Desktop 켜보세요